In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch import Tensor
from torchvision import transforms
import pandas as pd
import h5py

import matplotlib.pyplot as plt
import matplotlib.animation as animation

from IPython.display import HTML

import gc

# for random pruning
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device is {device}")

In [ ]:
# set hyperparameters
num_epochs = 200
batch_size = 128
learning_rate = 1e-3
weight_decay = 1e-4
data_source = "128x172" #or "new"
# for models
save_path = "/home/oltans/snn_development/DVS-Resnet/models/"

In [ ]:
# import and explore data
if data_source == "new":
    data_location = "/home/oltans/snn_development/datasets/DDD-20-single-recording/out_files/recording_bin50ms.hdf5"
    dvs_frame = np.array(h5py.File(data_location)["dvs_frame_224x224"])
else:
    data_location = f"/home/oltans/snn_development/datasets/DDD-20-single-recording/out_files/old_file/recording_bin50ms_{data_source}.hdf5"
    dvs_frame = np.array(h5py.File(data_location)[f"dvs_frame_{data_source}"])
headers = pd.DataFrame(np.array(h5py.File(data_location)))
print(f"The headers accessible are as such:\n{headers}")

print(f"Shape of dvs frame is {dvs_frame.shape}.")

steering_angles = np.array(h5py.File(data_location)["steering_wheel_angle"])
speeds = np.array(h5py.File(data_location)["vehicle_speed"])
# reshape to add second dimension
print(f"Number of ALL steering angles is {steering_angles.size}, with an average of {np.average(steering_angles)}.")
train_idxs = np.array(h5py.File(data_location)["train_idxs"])
test_idxs = np.array(h5py.File(data_location)["test_idxs"])
print(f"Originally here are {train_idxs.size} training indices, and {test_idxs.size} testing indices.")

train_speeds = speeds[train_idxs]

train_frames = dvs_frame[train_idxs]
test_frames = dvs_frame[test_idxs]
train_angles = steering_angles[train_idxs]
test_angles = steering_angles[test_idxs]
print(f"Thus, training frames are shaped as: {train_frames.shape} with testing frames shaped as: {test_frames.shape}.")
print(f"It logically follows that there are {train_angles.size} steering angles in training, and {test_angles.size} in testing.")

plt.figure(figsize=(15,7.5))
plt.plot(train_angles)
plt.title("Training Steering Angles Before Pruning")
plt.ylabel("Steering Angle (Degrees)")
plt.show()

# Perform preprocessing
# reduce the number of datapoints where the steering angle is low.
pruned_indices = []
for index, value in enumerate(train_angles):
    if abs(value) > 5 or random.random() < 0.3:
        pruned_indices.append(index)
print(f"The length of the += 5% pruned values is {len(pruned_indices)}.")

train_speeds = train_speeds[pruned_indices]
# Remove values where speed < 15kmh
pruned_indices = []
for index, value in enumerate(train_speeds):
    if value > 15:
        pruned_indices.append(index)

train_angles = train_angles[pruned_indices]
train_frames = train_frames[pruned_indices]
print(f"The length of the + 15kmh pruned values is {len(pruned_indices)}.")

# Normalise DVS Frames
# TODO NORMALISE THEM ALL DUMMY
train_min, train_max = train_frames.min(), train_frames.max()
print(f"Train frames had a min of {train_min}, max of {train_max} and an average of {np.average(train_frames)}.")
train_frames = (train_frames - train_min) / (train_max - train_min)
print(f"After normalisation, it has a min of {train_frames.min()}, max of {train_frames.max()} and an average of {np.average(train_frames)}.")

# Can have another step here to reduce outliers by removing anything more than 3 stdev away, we don't choose to here.

#Rename to fit convention
train_X, train_y = train_frames, train_angles
test_X, test_y = test_frames, test_angles

print(f"Following pruning, training frames are shaped as: {train_X.shape} with associated steering angles shape: {train_y.shape}.")

plt.figure(figsize=(15,7.5))
plt.plot(train_angles)
plt.title("Training Steering Angles After Pruning")
plt.ylabel("Steering Angle (Degrees)")
plt.show()

frames = [] # for storing the generated images
fig = plt.figure(figsize=(8, 8))
plt.title("Example DVS Frames")
for i in range(100):
    frames.append([plt.imshow(train_X[len(train_X) // 2 + i], cmap="gray", vmin=0, vmax=1.0, animated=True, origin="lower")])
ani = animation.ArtistAnimation(fig, frames, interval=50, blit=True, repeat_delay=1000)
display(HTML(ani.to_html5_video()))
plt.close(fig)

In [ ]:
class DDD20Dataset(Dataset):
    def __init__(self, data, target, transform=None, target_transform=None):
        self.data = data
        self.target = target
        self.transform = transform
        self.target_transform = target_transform
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        x = self.data[index]
        if self.transform:
            x = self.transform(x)
        
        y = self.target[index]
        if self.target_transform:
            y = self.target_transform(y)
        return x, y

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.float())
])

test_dataset = DDD20Dataset(test_X, Tensor(test_y), transform=transform)
train_dataset = DDD20Dataset(train_X, Tensor(train_y), transform=transform)

# load the data, can implement batching etc. here
train_loader = DataLoader(train_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride = 1, downsample = None):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Sequential(
                        nn.Conv2d(in_channels, out_channels, kernel_size = 3, stride = stride, padding = 1),
                        nn.BatchNorm2d(out_channels),
                        nn.ReLU())
        self.conv2 = nn.Sequential(
                        nn.Conv2d(out_channels, out_channels, kernel_size = 3, stride = 1, padding = 1),
                        nn.BatchNorm2d(out_channels))
        self.downsample = downsample
        self.relu = nn.ReLU()
        self.out_channels = out_channels

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.conv2(out)
        if self.downsample:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.conv2(out)
        if self.downsample:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes = 1):
        super(ResNet, self).__init__()
        self.inplanes = 64
        self.conv1 = nn.Sequential(
                        nn.Conv2d(1, 64, kernel_size = 7, stride = 2, padding = 3),
                        nn.BatchNorm2d(64),
                        nn.ReLU())
        self.maxpool = nn.MaxPool2d(kernel_size = 3, stride = 2, padding = 1)
        self.layer0 = self._make_layer(block, 64, layers[0], stride = 1)
        self.layer1 = self._make_layer(block, 128, layers[1], stride = 2)
        self.layer2 = self._make_layer(block, 256, layers[2], stride = 2)
        self.layer3 = self._make_layer(block, 512, layers[3], stride = 2)
        self.avgpool = nn.AvgPool2d(7, stride=1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes:

            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, kernel_size=1, stride=stride),
                nn.BatchNorm2d(planes),
            )
        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = x.squeeze(0)
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x

In [ ]:
model = ResNet(ResidualBlock, [3, 4, 6, 3]).to(device)

#Loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)  

#Train the model
total_step = len(train_loader)

train_losses = []
test_losses = []

total_step = len(train_loader)

for epoch in range(num_epochs):
    epoch_train_loss = 0
    for i, (dvs_images, steering_angle) in enumerate(train_loader):  
      #Move tensors to the configured device
      dvs_images = dvs_images.to(device)
      steering_angle = steering_angle.to(device)
      #Forward pass
      outputs = model(dvs_images.unsqueeze(0))
      loss = criterion(outputs, steering_angle.unsqueeze(1))
      #Backward and optimize
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()
      epoch_train_loss += loss.item()
      del dvs_images, steering_angle, outputs
      torch.cuda.empty_cache()
      gc.collect()

    # Average training loss for the epoch
    epoch_train_loss /= len(train_loader)
    train_losses.append(epoch_test_loss)

    #Validation
    epoch_test_loss = 0
    with torch.no_grad():
          for dvs_images, steering_angle in test_loader:
              dvs_images = dvs_images.to(device)
              steering_angle = steering_angle.to(device)
              outputs = model(dvs_images.unsqueeze(0))
              loss = criterion(outputs, steering_angle.unsqueeze(1))
              epoch_test_loss += loss.item()
              del dvs_images, steering_angle, outputs
          epoch_test_loss /= len(test_loader)
          if not ((epoch + 1) % 20):
              state = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': loss
            }
              torch.save(state, f"{save_path}e{epoch}_L{epoch_test_loss: .2f}.pth")
          test_losses.append(epoch_test_loss)
          if not ((epoch + 1) % 5):
            print(f"Epoch [{epoch + 1}/{num_epochs}], Train Loss: {epoch_train_loss: .4f} Test Loss: {epoch_test_loss: .4f}")
            
# Plotting the training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs + 1), train_losses, label='Training Loss')
plt.plot(range(1, num_epochs + 1), test_losses, label='Test Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (Degrees of Steering)')
plt.title('Training and Test Losses')
plt.legend()
plt.show()


In [ ]:
# Function to generate predictions in batches
def generate_predictions(model, data_X, data_y, batch_size, device, transform):
    model.eval()
    predictions = []
    ground_truth = []
    with torch.no_grad():
        for i in range(0, len(data_X), batch_size):
            batch_X = data_X[i:i+batch_size]
            batch_y = data_y[i:i+batch_size]
            transformed_batch_X = torch.stack([transform(img) for img in batch_X])
            batch_X_tensor = transformed_batch_X.to(device)
            batch_y_tensor = Tensor(batch_y).to(device)
            outputs = model(batch_X_tensor)
            predictions.extend(outputs.detach().cpu().numpy())
            ground_truth.extend(batch_y_tensor.cpu().numpy())
            del batch_X_tensor, batch_y_tensor, outputs
            torch.cuda.empty_cache()
            gc.collect()
    return predictions, ground_truth

# Generate predictions for training set
train_outputs_np, train_y_np = generate_predictions(model, train_X, train_y, batch_size, device, transform)

# Generate predictions for testing set
test_outputs_np, test_y_np = generate_predictions(model, test_X, test_y, batch_size, device, transform)

# Plotting
plt.figure(figsize=(20,10))

# Plot training set predictions
plt.subplot(2, 1, 1)
plt.plot(train_y, label="Ground Truth - Training")
plt.plot(train_outputs_np, label="Model Prediction - Training")
plt.title("Training Angles vs Model Prediction")
plt.xlabel("Time")
plt.ylabel("Steering Angle")
plt.legend()

# Plot testing set predictions
plt.subplot(2, 1, 2)
plt.plot(test_y, label="Ground Truth - Testing")
plt.plot(test_outputs_np, label="Model Prediction - Testing")
plt.title("Testing Angles vs Model Prediction")
plt.xlabel("Time")
plt.ylabel("Steering Angle")
plt.legend()

plt.tight_layout()
plt.show()